# Using NLP to Understand Construction Safety Failures from Accident Narratives[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yenlikgaisina-ux/construction-safety-nlp/blob/main/notebooks/construction_safety_nlp_analysis.ipynb)**Author:** Yenlik Gaisina  **Project:** Portfolio NLP Case Study  **Goal:** Apply Natural Language Processing techniques to OSHA construction accident narratives to surface recurring safety failure patterns, hazard categories, and recommendations.This notebook walks through a full NLP pipeline: data loading, cleaning, exploratory analysis, word frequency, word clouds, rule-based hazard categorisation, topic modelling with BERTopic, and LDA comparison. The end result is a set of business-ready safety recommendations.

## 1. Business Problem and Project AimConstruction is one of the highest-risk industries worldwide. Every year, thousands of accident narratives are filed with OSHA, but they remain mostly unread because reviewing each one manually is expensive and slow.**Business question:** Can we use NLP to automatically surface the most common failure patterns hidden in these narratives so that safety teams can act on them?**Project aim:**- Quantify the most frequent hazard types in construction accidents- Identify latent topics in the narratives using topic modelling- Translate insights into clear, prioritised safety recommendations

In [ ]:
# Install requirements (uncomment to run in Colab)# !pip install pandas numpy matplotlib nltk wordcloud bertopic sentence-transformers umap-learn hdbscan gensim pyLDAvis scikit-learn

## 2. Data Source**Primary dataset:** [OSHA Accident and Injury Data 2015-2017 on Kaggle](https://www.kaggle.com/datasets/ruqaiyaship/osha-accident-and-injury-data-1517)The dataset contains free-text accident narratives describing the events leading up to construction-site incidents. We do not redistribute the raw data  please download it directly from Kaggle and place it in `data/raw/`.

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport reimport warningswarnings.filterwarnings('ignore')# Configure displaypd.set_option('display.max_colwidth', 200)plt.rcParams['figure.figsize'] = (10, 6)

## 3. Data Loading and Initial Inspection

In [ ]:
# Load the datasetdf = pd.read_csv('../data/raw/osha.csv', encoding='latin-1')print(f'Dataset shape: {df.shape}')df.head()

In [ ]:
# Inspect columns and data typesprint(df.info())print('\nMissing values per column:')print(df.isnull().sum())

## 4. Data CleaningWe focus on the narrative column. Steps:- Drop rows where the narrative is missing- Strip whitespace- Remove duplicates

In [ ]:
# Identify the narrative column (typical column name in this dataset)narrative_col = 'Abstract Text' if 'Abstract Text' in df.columns else df.columns[-1]print(f'Using column: {narrative_col}')df = df.dropna(subset=[narrative_col]).copy()df[narrative_col] = df[narrative_col].astype(str).str.strip()df = df.drop_duplicates(subset=[narrative_col])print(f'After cleaning: {df.shape}')

## 5. Text PreprocessingStandard NLP pipeline:- Lowercase- Remove punctuation and digits- Tokenise- Remove stopwords- Lemmatise

In [ ]:
import nltknltk.download('stopwords', quiet=True)nltk.download('wordnet', quiet=True)nltk.download('punkt', quiet=True)from nltk.corpus import stopwordsfrom nltk.stem import WordNetLemmatizerfrom nltk.tokenize import word_tokenizestop_words = set(stopwords.words('english'))extra_stops = {'employee', 'worker', 'one', 'two', 'mr', 'feet', 'foot', 'inch', 'approximately'}stop_words.update(extra_stops)lemmatizer = WordNetLemmatizer()def preprocess(text):    text = text.lower()    text = re.sub(r'[^a-z\s]', ' ', text)    tokens = word_tokenize(text)    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]    return ' '.join(tokens)df['clean_text'] = df[narrative_col].apply(preprocess)df[[narrative_col, 'clean_text']].head(3)

## 6. Exploratory Data Analysis

In [ ]:
# Narrative length distributiondf['word_count'] = df['clean_text'].str.split().str.len()df['word_count'].describe()

In [ ]:
# Plot word count distributionplt.figure(figsize=(10, 5))df['word_count'].hist(bins=50, color='#1f77b4', edgecolor='white')plt.xlabel('Word count (cleaned)')plt.ylabel('Number of narratives')plt.title('Distribution of narrative length')plt.tight_layout()plt.savefig('../visuals/narrative_length.png', dpi=150)plt.show()

## 7. Word Frequency Analysis

In [ ]:
from collections import Counterall_words = ' '.join(df['clean_text']).split()word_freq = Counter(all_words)top20 = word_freq.most_common(20)print(top20)

In [ ]:
# Plot top 20 wordswords, counts = zip(*top20)plt.figure(figsize=(12, 6))plt.barh(words[::-1], counts[::-1], color='#ff7f0e')plt.xlabel('Frequency')plt.title('Top 20 most frequent words in construction accident narratives')plt.tight_layout()plt.savefig('../visuals/top_20_words.png', dpi=150)plt.show()

## 8. Word Cloud

In [ ]:
from wordcloud import WordCloudwc = WordCloud(width=1200, height=600, background_color='white',               colormap='viridis', max_words=150).generate(' '.join(df['clean_text']))plt.figure(figsize=(15, 7))plt.imshow(wc, interpolation='bilinear')plt.axis('off')plt.title('Construction accident narratives  word cloud', fontsize=14)plt.tight_layout()plt.savefig('../visuals/accident_narratives_wordcloud.png', dpi=150, bbox_inches='tight')plt.show()

## 9. Rule-Based Hazard Keyword GroupingBefore topic modelling, we use a rule-based approach grounded in OSHA's Fatal Four hazards plus other common construction risks.

In [ ]:
hazard_keywords = {    'Falls':         ['fall', 'fell', 'scaffold', 'ladder', 'roof', 'height', 'platform'],    'Struck-by':     ['struck', 'hit', 'crushed', 'object', 'falling'],    'Caught-in':     ['caught', 'trapped', 'pinned', 'trench', 'collapse'],    'Electrocution': ['electric', 'shock', 'voltage', 'wire', 'electrocuted'],    'Machinery':     ['machine', 'forklift', 'crane', 'saw', 'equipment', 'truck'],    'Chemical':      ['chemical', 'fume', 'inhaled', 'burn', 'acid', 'solvent'],    'Fire/Explosion':['fire', 'explosion', 'flame', 'ignited', 'gas']}def classify(text):    cats = []    for cat, kws in hazard_keywords.items():        if any(k in text for k in kws):            cats.append(cat)    return cats if cats else ['Other']df['hazards'] = df['clean_text'].apply(classify)hazard_counts = Counter(h for hs in df['hazards'] for h in hs)print(hazard_counts.most_common())

In [ ]:
# Plot hazard category distributioncats, counts = zip(*hazard_counts.most_common())plt.figure(figsize=(11, 6))plt.bar(cats, counts, color=['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd','#8c564b','#e377c2','#7f7f7f'])plt.xlabel('Hazard category')plt.ylabel('Number of narratives mentioning hazard')plt.title('Construction accident hazard categories (rule-based)')plt.xticks(rotation=20)plt.tight_layout()plt.savefig('../visuals/hazard_categories_bar_chart.png', dpi=150)plt.show()

## 10. Topic Modelling with BERTopicBERTopic uses sentence transformers + UMAP + HDBSCAN to discover semantically coherent topics that go beyond keyword overlap.

In [ ]:
from bertopic import BERTopicfrom sentence_transformers import SentenceTransformer# Sample for speed if dataset is largesample = df['clean_text'].sample(min(3000, len(df)), random_state=42).tolist()embedding_model = SentenceTransformer('all-MiniLM-L6-v2')topic_model = BERTopic(embedding_model=embedding_model, min_topic_size=15, verbose=True)topics, probs = topic_model.fit_transform(sample)topic_model.get_topic_info().head(10)

In [ ]:
# Visualise top topicsfig = topic_model.visualize_barchart(top_n_topics=8, n_words=8)fig.write_image('../visuals/bertopic_topic_barchart.png', width=1200, height=700)fig

## 11. LDA ComparisonWe benchmark BERTopic against classic LDA to confirm the themes are robust.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizerfrom sklearn.decomposition import LatentDirichletAllocationvectorizer = CountVectorizer(max_features=2000, min_df=5, max_df=0.9)X = vectorizer.fit_transform(df['clean_text'])lda = LatentDirichletAllocation(n_components=6, random_state=42, learning_method='online')lda.fit(X)feature_names = vectorizer.get_feature_names_out()lda_topics = []for idx, topic in enumerate(lda.components_):    top_words = [feature_names[i] for i in topic.argsort()[-8:][::-1]]    lda_topics.append(top_words)    print(f'Topic {idx}: {", ".join(top_words)}')

In [ ]:
# Plot LDA topics summaryfig, ax = plt.subplots(figsize=(12, 7))ax.axis('off')table_data = [[f'Topic {i}'] + words for i, words in enumerate(lda_topics)]table = ax.table(cellText=table_data,                 colLabels=['Topic']+[f'W{i+1}' for i in range(8)],                 loc='center', cellLoc='left')table.auto_set_font_size(False)table.set_fontsize(10)table.scale(1, 1.8)plt.title('LDA topics summary (6 topics  top 8 words)', fontsize=13, pad=20)plt.tight_layout()plt.savefig('../visuals/lda_topics_summary.png', dpi=150, bbox_inches='tight')plt.show()

## 12. Key Findings1. **Falls dominate.** Roughly one in three narratives mention falls from ladders, scaffolds, or roofs.2. **Struck-by and caught-in events are the second tier**, often involving machinery or trenches.3. **BERTopic surfaces fine-grained themes** that rule-based grouping misses, e.g. electrical work near power lines, confined-space rescues, and forklift tip-overs.4. **LDA broadly agrees** with BERTopic's themes, increasing confidence in the patterns.

## 13. Safety RecommendationsBased on the NLP findings, the top five interventions for construction firms:1. **Fall protection audits** on scaffolds, ladders and roof edges every quarter.2. **Mandatory struck-by training** for ground crews near heavy equipment.3. **Trench shoring sign-off** for any excavation deeper than 5 ft.4. **Lock-out / tag-out enforcement** before energised electrical work.5. **Toolbox talks targeted to the BERTopic themes** so safety briefings reflect the actual failure patterns.

## 14. Limitations- The OSHA dataset is biased toward serious / reportable incidents and may under-represent near-misses.- Narratives are written by investigators, so phrasing is consistent but may bury root causes.- Topic models surface correlations, not causation.- The 20152017 window pre-dates several recent regulatory changes.

## 15. ConclusionNLP turns a stack of unread incident reports into a quantified, prioritised safety dashboard. By combining rule-based hazard tagging with BERTopic and LDA, we can identify recurring failure patterns at scale and translate them into actionable interventions.**Next steps:**- Run on the latest 20182024 OSHA data- Cross-reference with injury severity to weight recommendations- Build a simple Streamlit dashboard for safety officers